# Week 6: Zero-shot vs fine-tuned comparison + Gradio UI

Compare the same test set with **zero-shot** (frontier LLM) and **fine-tuned** (GPT-4.1-nano style) predictors, then try both in a **Gradio** app.

In [1]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv
from huggingface_hub import login
from sklearn.metrics import mean_squared_error, r2_score
import pandas as pd

# Ensure week6 is on path so pricer is found
_cwd = Path.cwd()
if (_cwd / "week6" / "pricer").exists():
    sys.path.insert(0, str(_cwd / "week6"))
elif (_cwd / "pricer").exists():
    sys.path.insert(0, str(_cwd))
else:
    sys.path.insert(0, str(_cwd.parent.parent.parent))  # from "week 6 exercise" folder

from pricer.items import Item
from tqdm.notebook import tqdm

load_dotenv(override=True)
login(os.environ["HF_TOKEN"], add_to_git_credential=True)

ModuleNotFoundError: No module named 'pricer'

In [ ]:
username = "ed-donner"
dataset = f"{username}/items_lite"  # or items_full
train, val, test = Item.from_hub(dataset)
print(f"Train: {len(train):,}, Val: {len(val):,}, Test: {len(test):,}")

## Predictors

Zero-shot uses LiteLLM; fine-tuned uses your OpenAI fine-tuned model. Set `WEEK6_FINE_TUNED_MODEL` or pass the model name after a run.

In [ ]:
import sys
sys.path.insert(0, ".")
from pricers import zero_shot_predict, fine_tuned_predict

# Set after a fine-tune job (or export WEEK6_FINE_TUNED_MODEL). Leave blank to skip fine-tuned in comparison.
FINE_TUNED_MODEL_NAME = os.environ.get("WEEK6_FINE_TUNED_MODEL", "")  # e.g. "ft:gpt-4.1-nano:..."

def zero_shot_pricer(item):
    return zero_shot_predict(item)

def fine_tuned_pricer(item):
    return fine_tuned_predict(item, model=FINE_TUNED_MODEL_NAME or None)

## Run comparison on the same test subset

Both models are evaluated on the same `size` items; metrics are printed side by side.

In [ ]:
COMPARE_SIZE = 50  # increase for a more stable comparison (more API calls)
subset = test[:COMPARE_SIZE]

def run_metrics(predict_fn, data, name):
    truths = [item.price for item in data]
    guesses = []
    for item in tqdm(data, desc=name):
        try:
            g = predict_fn(item)
        except Exception as e:
            g = 0.0
        guesses.append(g)
    errors = [abs(g - t) for g, t in zip(guesses, truths)]
    mae = sum(errors) / len(errors)
    mse = mean_squared_error(truths, guesses)
    r2 = r2_score(truths, guesses) * 100
    return {"MAE": mae, "MSE": mse, "R² %": r2, "guesses": guesses, "truths": truths}

metrics_zs = run_metrics(zero_shot_pricer, subset, "Zero-shot")
metrics_ft = run_metrics(fine_tuned_pricer, subset, "Fine-tuned") if FINE_TUNED_MODEL_NAME else None

In [ ]:
rows = [
    {"Model": "Zero-shot", "MAE $": f"{metrics_zs['MAE']:.2f}", "MSE": f"{metrics_zs['MSE']:,.0f}", "R² %": f"{metrics_zs['R² %']:.1f}"}
]
if metrics_ft:
    rows.append({"Model": "Fine-tuned", "MAE $": f"{metrics_ft['MAE']:.2f}", "MSE": f"{metrics_ft['MSE']:,.0f}", "R² %": f"{metrics_ft['R² %']:.1f}"})
pd.DataFrame(rows)

## Gradio UI

Paste a product description and choose **Zero-shot** or **Fine-tuned** to get a price prediction.

In [ ]:
import gradio as gr

def predict_price(description: str, model_choice: str):
    if not description or not description.strip():
        return "Enter a product description."
    try:
        if model_choice == "Fine-tuned":
            if not FINE_TUNED_MODEL_NAME:
                return "Set FINE_TUNED_MODEL_NAME (or WEEK6_FINE_TUNED_MODEL) to use the fine-tuned model."
            price = fine_tuned_predict(description, model=FINE_TUNED_MODEL_NAME)
        else:
            price = zero_shot_predict(description)
        return f"${price:,.2f}"
    except Exception as e:
        return f"Error: {e}"

demo = gr.Interface(
    fn=predict_price,
    inputs=[
        gr.Textbox(placeholder="Paste product title/description here...", label="Product description", lines=4),
        gr.Radio(choices=["Zero-shot", "Fine-tuned"], value="Zero-shot", label="Model"),
    ],
    outputs=gr.Textbox(label="Predicted price"),
    title="Price is Right — Week 6",
    description="Predict product price from description using zero-shot or fine-tuned model.",
)
demo.launch()